In [ ]:
import pandas as pd
from gensim.models import Word2Vec
import gensim
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download('punkt_tab')
import warnings
import re
import faiss

import numpy as np
warnings.filterwarnings(action='ignore')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Data prepration

In [3]:

df = pd.read_json("data/processed/cleaned_data.json")

print(df.head())
print(df.columns)

                                                   0
0  socorro county new mexico infobox us county co...
1  david coventry infobox writer information see ...
2  clelio darida fileclelio daridajpgthumbclelio ...
3  hcone hcone britains largest residential insti...
4  mighty morphin power rangers infobox televisio...
RangeIndex(start=0, stop=1, step=1)


In [4]:
documents = df[0].tolist()

In [5]:
print(type(documents))   # should be list
print(documents[:3])     # preview
print(len(documents))    # number of docs

<class 'list'>
['socorro county new mexico infobox us county county socorro county state new mexico flag seal founded year 1852 founded date january 9 seat wl socorro largest city wl socorro areatotalsqmi 6649 arealandsqmi 6647 areawatersqmi 21 area percentage 003 census yr 2020 pop 16595 densitysqmi auto web wwwsocorrocountynet ex image socorro county new mexico court housejpg ex image cap socorro county courthouse socorro district 2nd time zone mountain socorro county list counties new mexicocounty us state new mexico 2020 united states census2020 census population 16595ref nameqfcite webtitlequickfacts socorro county new mexicopublisherunited states census bureauurlhttpswwwcensusgovquickfactssocorrocountynewmexicoaccessdatejanuary 17 2024ref county seat socorro new mexicosocorroref namegr6cite weburlhttpwwwnacoorgcountiespagesfindacountyaspxaccessdatejune 7 2011titlefind countypublishernational association countiesref references reflist new mexico socorro county new mexico category1

In [6]:
print(documents[:3])

['socorro county new mexico infobox us county county socorro county state new mexico flag seal founded year 1852 founded date january 9 seat wl socorro largest city wl socorro areatotalsqmi 6649 arealandsqmi 6647 areawatersqmi 21 area percentage 003 census yr 2020 pop 16595 densitysqmi auto web wwwsocorrocountynet ex image socorro county new mexico court housejpg ex image cap socorro county courthouse socorro district 2nd time zone mountain socorro county list counties new mexicocounty us state new mexico 2020 united states census2020 census population 16595ref nameqfcite webtitlequickfacts socorro county new mexicopublisherunited states census bureauurlhttpswwwcensusgovquickfactssocorrocountynewmexicoaccessdatejanuary 17 2024ref county seat socorro new mexicosocorroref namegr6cite weburlhttpwwwnacoorgcountiespagesfindacountyaspxaccessdatejune 7 2011titlefind countypublishernational association countiesref references reflist new mexico socorro county new mexico category1852 establishme

## Build embeddings

In [22]:
data = []
for doc in documents:
    for i in sent_tokenize(doc):
        temp = []

        for j in word_tokenize(i):
            temp.append(j.lower())

        data.append(temp)

In [23]:
model1 = gensim.models.Word2Vec(data, min_count=1,
                                vector_size=100, window=5)

In [24]:
import numpy as np

def get_vector(tokens, model):
    vectors = []

    for w in tokens:
        if w in model.wv:
            vectors.append(model.wv[w])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
#this cell will be ignored during discussion
chunk_embeddings = []

for tokens in data:
    vec = get_vector(tokens, model1)
    chunk_embeddings.append(vec)

chunk_embeddings = np.array(chunk_embeddings)
np.save("embeddings.npy", chunk_embeddings)
print(chunk_embeddings.shape)

(2000, 100)


In [ ]:
np.save("embeddings.npy", chunk_embeddings)

In [32]:
chunk_embeddings = np.load("embeddings.npy")
print(chunk_embeddings.shape)

(2000, 100)


## FAISS index

In [33]:
dim = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dim)
index.add(chunk_embeddings)

In [34]:
def search(query, model, index, k=5):
    tokens = query.lower().split()

    vec = get_vector(tokens, model)
    vec = np.array([vec])

    distances, indices = index.search(vec, k)

    return indices[0], distances[0]

In [35]:
with open('queries.txt', 'r') as f:
    queries = [line.strip() for line in f.readlines()]

print("Queries loaded:")
for i, query in enumerate(queries, 1):
    print(f"{i}. {query}")
print(f"\nTotal queries: {len(queries)}")

Queries loaded:
1. food delivery problem
2. climate change effects
3. machine learning applications
4. health benefits of exercise
5. economic inflation causes

Total queries: 5


In [36]:
for query in queries:
    indices, distances = search(query, model1, index)
    print(f"\nQuery: {query}")
    print("Top 5 results:")
    for idx, dist in zip(indices, distances):
        print(f"Distance: {dist:.4f}, Text: {documents[idx][:100]}...")


Query: food delivery problem
Top 5 results:
Distance: 0.0237, Text: plywood filebirke multiplexjpgthumbbirch plywood plywood manufactured wood made gluegluing several t...
Distance: 0.0328, Text: chronicle filebern burgerbibliothek cod 596 f 74v – anonymum byzantinum chroniconjpgthumba manuscrip...
Distance: 0.0350, Text: elaine corbenic elaine corbenic legendary character king arthur stories lancelot parents galahad dau...
Distance: 0.0379, Text: potassium chromate filekaliumchromatjpgthumbpotassium chromate potassium chromate chemical compound ...
Distance: 0.0412, Text: cohesion cohesion may mean enwiktionarycohesion cohesion chemistry intermolecular attraction likemol...

Query: climate change effects
Top 5 results:
Distance: 0.7107, Text: full moon party full moon partyref namefull moon party dateshttpsphangantravelfullmoonparty full moo...
Distance: 0.9309, Text: conservative party uk infobox political party name conservative unionist party logo fileconservative...
Distance: 1.2